# Imports

In [31]:
# Langchain modules
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import HumanMessage

# Helper modules
from dotenv import load_dotenv

# Web Search Imports
from typing import Dict, Any
from tavily import TavilyClient

# Cleanup module
from pprint import pprint

In [13]:
load_dotenv()

True

## Basic Tool Use

Simply put: tools are functions the agent can call

Anatomy of a tool:

- @tool decorator from langchain.tools (from langchain.tools import tool)
- Name of the tool --> needs to explain what the tool does
    - Alternatively can add a string inside the @tool call (e.g. @tool("does something useful")) to make it even more explicit to the agent what the tool does
- Docstring that explains what the tool (funcion) does
    - Can override this with the description optional argument in the decorator

In [7]:
@tool
def square_a_number(x: float) -> float:
    """Calculate the square of a number"""
    return x * x

Agents invoke tools with the proper arguments like the example below:

In [8]:
square_a_number.invoke({"x": 12})

144.0

Defining an agent with a tool using the create_agent class requires a few core building blocks:
- model --> simple and the same as above
- tools --> list of tools for the agent to use
- system_prompt --> instructions to the agent telling it what it is, its goals and how to solve problems

In [9]:
agent_with_tool_sys_prompt = """
You really like to square a number someone give you. 
When someone asks about a number, find its square 
and tell them why it's meaningful to you.
"""

agent_with_tool = create_agent(
    model='claude-haiku-4-5',
    tools=[square_a_number],
    system_prompt=agent_with_tool_sys_prompt
)

Use the .invoke() method of the create_agent class to take action. Pass the message in a dictionary in the invoke call. It returns the result as a separate object.

In [10]:
my_question = HumanMessage(content="What is the square of 15?")

agent_response = agent_with_tool.invoke({'messages': my_question})

In [ ]:
agent_response

The output from the tool is returned as a different object: ToolMessage

## More Useful Tool: Web Search

In [14]:
# setting up a tavily client to search the web
tavily_client = TavilyClient()

# Creating a tool for the agent to use to search the web, can be called standalone
@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for specific info"""

    return tavily_client.search(query)

In [15]:
web_search.invoke("How many people live on Elephant Island?")

{'query': 'How many people live on Elephant Island?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://poseidonexpeditions.com/about/articles/elephant-island-antarctica-interesting-facts',
   'title': 'Elephant Island, Antarctica - What you need to know | Poseidon Expeditions',
   'content': 'There is no human population on Elephant Island. It’s a barren island with no significant flora. The island does not offer safe anchorage, which historically prevented permanent human settlement on the island. It is a small and very remote island in the Southern Ocean, located around 240 km (150 miles) off the Antarctic Peninsula, in the outer reaches of the South Shetland Islands. On its rocky shores, you can see wildlife such as seabirds, chinstrap penguins, and occasionally Weddell seals and [...] ### Can you visit Elephant Island?\n\nIt is difficult to visit! Bleak, inhospitable, and with its rugged and rocky shores, it is inaccessible and exposed to 

In [25]:
web_search_agent = create_agent(
    model='claude-haiku-4-5',
    tools=[web_search],
    system_prompt="""You are a research assistant for a project about
    Ernest Shackleton's ill-fated adventure on The Endurance. Use the web
    to look up information to help the researrcher in learning more information
    about the survival story."""
)

In [22]:
my_web_message = HumanMessage(content="What is the current population of Elephant Island?")

In [26]:
web_agent_result = web_search_agent.invoke({'messages': my_web_message})

In [27]:
web_agent_result

{'messages': [HumanMessage(content='What is the current population of Elephant Island?', additional_kwargs={}, response_metadata={}, id='e5d20393-df82-426f-9431-7fc400ad84d9'),
  AIMessage(content=[{'id': 'toolu_01CJxLtiK1DvinMDJXsqcmP5', 'caller': {'type': 'direct'}, 'input': {'query': 'Elephant Island current population'}, 'name': 'web_search', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CdsK4wo6cyuheXtVrChbH', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 622, 'output_tokens': 58, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--019fe6de-cbaa-72f0-8cfc-47ef8fee5d2a-0', tool_

In [32]:
pprint(web_agent_result['messages'][-1].content)

('Based on the search results, **Elephant Island has no human population**. '
 'The island is uninhabited.\n'
 '\n'
 "This is particularly relevant to your research on Ernest Shackleton's "
 'Endurance expedition! In fact, Elephant Island holds an important place in '
 'that survival story. After the Endurance was crushed by ice in 1915, '
 'Shackleton and his crew eventually made their way to Elephant Island, which '
 'was the first land they reached after months at sea. While the island itself '
 'is barren, inhospitable, and has no permanent human settlement, it served as '
 "a crucial refuge for Shackleton's men while Shackleton and a small group "
 'attempted the famous boat journey to South Georgia to get rescue.\n'
 '\n'
 'The island does host wildlife, including chinstrap and gentoo penguins, and '
 'occasionally Weddell seals and elephant seals—but no people.')
